In [1]:
# =========================================
# 1. IMPORTS
# =========================================
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    ndcg_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
# =========================================
# 2. PATH SETUP
# =========================================
DATA_DIR = Path(r"C:\D drive\item_recommendation\data")

RANKER_FILE = DATA_DIR / "ranker_training_dataset.csv"
MODEL_DIR = DATA_DIR / "model_outputs"

MODEL_FILE = MODEL_DIR / "xgboost_ranker_model.json"
FEATURE_FILE = MODEL_DIR / "ranker_feature_columns.json"

print("Ranker dataset exists:", RANKER_FILE.exists())
print("Model file exists:", MODEL_FILE.exists())
print("Feature file exists:", FEATURE_FILE.exists())
print("Model dir:", MODEL_DIR)

Ranker dataset exists: True
Model file exists: True
Feature file exists: True
Model dir: C:\D drive\item_recommendation\data\model_outputs


In [3]:
# =========================================
# 3. LOAD DATASET
# =========================================
ranker_df = pd.read_csv(RANKER_FILE)
ranker_df.columns = [c.strip() for c in ranker_df.columns]

print("Shape:", ranker_df.shape)
print("Columns:")
print(ranker_df.columns.tolist())

display(ranker_df.head())

Shape: (449680, 16)
Columns:
['customerId', 'candidate_item_id', 'candidate_category', 'basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'season', 'isHoliday', 'isFestival', 'timeSlot', 'weekOfMonth', 'label', 'group_id']


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,1
1,23561,952,Pantry-Rice,2,26.0,3.0,4.0,0.0,0.979893,Winter,0,0,Morning,1,0,1
2,23561,26615,Instant-Food,2,0.0,5.0,1.0,3.0,0.997745,Winter,0,0,Morning,1,0,1
3,23561,14034,Protein-Egg,2,29.0,4.0,7.0,2.0,0.998886,Winter,0,0,Morning,1,0,1
4,23561,2030,Pantry-Rice,2,23.5,3.0,2.0,3.0,0.979893,Winter,0,0,Morning,1,0,1


In [4]:
# =========================================
# 4. BASIC CLEANING
# =========================================
required_cols = [
    "group_id",
    "label",
    "candidate_item_id",
    "candidate_category",
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "season",
    "isHoliday",
    "isFestival",
    "timeSlot",
    "weekOfMonth"
]

missing_cols = [c for c in required_cols if c not in ranker_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

ranker_df["candidate_category"] = ranker_df["candidate_category"].fillna("Unknown").astype(str)
ranker_df["season"] = ranker_df["season"].fillna("Unknown").astype(str)
ranker_df["timeSlot"] = ranker_df["timeSlot"].fillna("Unknown").astype(str)

numeric_cols = [
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "isHoliday",
    "isFestival",
    "weekOfMonth",
    "label"
]

for col in numeric_cols:
    ranker_df[col] = pd.to_numeric(ranker_df[col], errors="coerce")

ranker_df = ranker_df.dropna(subset=numeric_cols).copy()

print("Cleaned shape:", ranker_df.shape)
display(ranker_df.head())

Cleaned shape: (449680, 16)


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,1
1,23561,952,Pantry-Rice,2,26.0,3.0,4.0,0.0,0.979893,Winter,0,0,Morning,1,0,1
2,23561,26615,Instant-Food,2,0.0,5.0,1.0,3.0,0.997745,Winter,0,0,Morning,1,0,1
3,23561,14034,Protein-Egg,2,29.0,4.0,7.0,2.0,0.998886,Winter,0,0,Morning,1,0,1
4,23561,2030,Pantry-Rice,2,23.5,3.0,2.0,3.0,0.979893,Winter,0,0,Morning,1,0,1


In [5]:
# =========================================
# 5. GROUP WISE TRAIN VALID TEST SPLIT
# =========================================
all_groups = ranker_df["group_id"].drop_duplicates().tolist()

random.seed(42)
random.shuffle(all_groups)

n_groups = len(all_groups)
train_end = int(n_groups * 0.80)
valid_end = int(n_groups * 0.90)

train_groups = set(all_groups[:train_end])
valid_groups = set(all_groups[train_end:valid_end])
test_groups = set(all_groups[valid_end:])

train_df = ranker_df[ranker_df["group_id"].isin(train_groups)].copy()
valid_df = ranker_df[ranker_df["group_id"].isin(valid_groups)].copy()
test_df = ranker_df[ranker_df["group_id"].isin(test_groups)].copy()

print("Train shape:", train_df.shape, "| groups:", train_df["group_id"].nunique())
print("Valid shape:", valid_df.shape, "| groups:", valid_df["group_id"].nunique())
print("Test shape :", test_df.shape, "| groups:", test_df["group_id"].nunique())

Train shape: (359744, 16) | groups: 32704
Valid shape: (44968, 16) | groups: 4088
Test shape : (44968, 16) | groups: 4088


In [6]:
# =========================================
# 6. FEATURE COLUMNS
# =========================================
feature_cols = [
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "isHoliday",
    "isFestival",
    "weekOfMonth",
    "season",
    "timeSlot",
    "candidate_category"
]

target_col = "label"
group_col = "group_id"

print("Feature columns:")
print(feature_cols)

Feature columns:
['basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'isHoliday', 'isFestival', 'weekOfMonth', 'season', 'timeSlot', 'candidate_category']


In [7]:
# =========================================
# 7. ONE HOT ENCODING
# =========================================
X_train_raw = train_df[feature_cols].copy()
X_valid_raw = valid_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()

X_train = pd.get_dummies(X_train_raw, columns=["season", "timeSlot", "candidate_category"])
X_valid = pd.get_dummies(X_valid_raw, columns=["season", "timeSlot", "candidate_category"])
X_test = pd.get_dummies(X_test_raw, columns=["season", "timeSlot", "candidate_category"])

X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df[target_col].astype(int).values
y_valid = valid_df[target_col].astype(int).values
y_test = test_df[target_col].astype(int).values

qid_train = train_df[group_col].values
qid_valid = valid_df[group_col].values
qid_test = test_df[group_col].values

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test shape :", X_test.shape)

X_train shape: (359744, 59)
X_valid shape: (44968, 59)
X_test shape : (44968, 59)


In [8]:
# =========================================
# 8. SORT BY GROUP ID
# =========================================
def sort_by_qid(X, y, qid):
    temp = X.copy()
    temp["_label"] = y
    temp["_qid"] = qid

    temp = temp.sort_values("_qid").reset_index(drop=True)

    y_sorted = temp["_label"].values
    qid_sorted = temp["_qid"].values
    X_sorted = temp.drop(columns=["_label", "_qid"])

    return X_sorted, y_sorted, qid_sorted

X_train_sorted, y_train_sorted, qid_train_sorted = sort_by_qid(X_train, y_train, qid_train)
X_valid_sorted, y_valid_sorted, qid_valid_sorted = sort_by_qid(X_valid, y_valid, qid_valid)
X_test_sorted, y_test_sorted, qid_test_sorted = sort_by_qid(X_test, y_test, qid_test)

print("Sorted train shape:", X_train_sorted.shape)
print("Sorted valid shape:", X_valid_sorted.shape)
print("Sorted test shape :", X_test_sorted.shape)

Sorted train shape: (359744, 59)
Sorted valid shape: (44968, 59)
Sorted test shape : (44968, 59)


In [9]:
# =========================================
# 9. LOAD TRAINED MODEL
# =========================================
ranker = xgb.XGBRanker()
ranker.load_model(MODEL_FILE)

with open(FEATURE_FILE, "r", encoding="utf-8") as f:
    trained_feature_columns = json.load(f)

print("Loaded model successfully")
print("Number of trained feature columns:", len(trained_feature_columns))

Loaded model successfully
Number of trained feature columns: 59


In [10]:
# =========================================
# 10. ALIGN WITH TRAINED FEATURES
# =========================================
X_train_sorted = X_train_sorted.reindex(columns=trained_feature_columns, fill_value=0)
X_valid_sorted = X_valid_sorted.reindex(columns=trained_feature_columns, fill_value=0)
X_test_sorted = X_test_sorted.reindex(columns=trained_feature_columns, fill_value=0)

print("Aligned train shape:", X_train_sorted.shape)
print("Aligned valid shape:", X_valid_sorted.shape)
print("Aligned test shape :", X_test_sorted.shape)

Aligned train shape: (359744, 59)
Aligned valid shape: (44968, 59)
Aligned test shape : (44968, 59)


In [11]:
# =========================================
# 11. PREDICT SCORES
# =========================================
train_meta = train_df.copy().sort_values(group_col).reset_index(drop=True)
valid_meta = valid_df.copy().sort_values(group_col).reset_index(drop=True)
test_meta = test_df.copy().sort_values(group_col).reset_index(drop=True)

train_meta["pred_score"] = ranker.predict(X_train_sorted)
valid_meta["pred_score"] = ranker.predict(X_valid_sorted)
test_meta["pred_score"] = ranker.predict(X_test_sorted)

display(test_meta.head())

,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id,pred_score
0,23494,708,Spices-Cooking,6,14.166667,94.0,2.0,3.0,0.979264,Winter,0,0,Afternoon,1,1,27,0.107744
1,23494,13352,Protein-Egg,6,22.000000,42.0,3.0,25.0,0.997620,Winter,0,0,Afternoon,1,0,27,-0.099650
2,23494,6815,Beverage-Hot,6,5.666667,16.0,4.0,2.0,0.998512,Winter,0,0,Afternoon,1,0,27,-0.238307
3,23494,15279,Veg-Cooking,6,6.833333,47.0,3.0,5.0,0.987591,Winter,0,0,Afternoon,1,0,27,0.728479
4,23494,13958,Fish-Fresh,6,14.500000,106.0,2.0,31.0,0.988419,Winter,0,0,Afternoon,1,0,27,0.013488


In [12]:
# =========================================
# 12. RANKING METRIC HELPERS
# =========================================
def precision_at_k(y_true, y_score, k=5):
    order = np.argsort(y_score)[::-1][:k]
    topk_true = np.array(y_true)[order]
    return float(np.sum(topk_true)) / float(k)

def recall_at_k(y_true, y_score, k=5):
    y_true = np.array(y_true)
    total_relevant = np.sum(y_true)

    if total_relevant == 0:
        return 0.0

    order = np.argsort(y_score)[::-1][:k]
    topk_true = y_true[order]
    return float(np.sum(topk_true)) / float(total_relevant)

def ndcg_at_k(y_true, y_score, k=5):
    y_true = np.array(y_true).reshape(1, -1)
    y_score = np.array(y_score).reshape(1, -1)
    return float(ndcg_score(y_true, y_score, k=k))

def evaluate_grouped_ranking(df_scored, group_col="group_id", label_col="label", score_col="pred_score", k=5):
    precision_scores = []
    recall_scores = []
    ndcg_scores = []

    for gid, grp in df_scored.groupby(group_col):
        y_true = grp[label_col].values
        y_score = grp[score_col].values

        precision_scores.append(precision_at_k(y_true, y_score, k=k))
        recall_scores.append(recall_at_k(y_true, y_score, k=k))
        ndcg_scores.append(ndcg_at_k(y_true, y_score, k=k))

    return {
        f"Precision@{k}": float(np.mean(precision_scores)),
        f"Recall@{k}": float(np.mean(recall_scores)),
        f"NDCG@{k}": float(np.mean(ndcg_scores))
    }

In [13]:
# =========================================
# 13. AUXILIARY CLASSIFICATION METRIC HELPERS
# =========================================
def build_top1_binary_view(df_scored, group_col="group_id", label_col="label", score_col="pred_score"):
    rows = []

    for gid, grp in df_scored.groupby(group_col):
        grp = grp.copy().sort_values(score_col, ascending=False).reset_index(drop=True)
        grp["pred_label_top1"] = 0
        if len(grp) > 0:
            grp.loc[0, "pred_label_top1"] = 1
        rows.append(grp)

    return pd.concat(rows, axis=0).reset_index(drop=True)

def classification_metrics(df_view):
    y_true = df_view["label"].astype(int).values
    y_pred = df_view["pred_label_top1"].astype(int).values

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0))
    }

In [14]:
# =========================================
# 14. BUILD AUXILIARY CLASSIFICATION VIEW
# =========================================
train_cls = build_top1_binary_view(train_meta)
valid_cls = build_top1_binary_view(valid_meta)
test_cls = build_top1_binary_view(test_meta)

print("Train top1 rows:", train_cls.shape)
print("Valid top1 rows:", valid_cls.shape)
print("Test top1 rows :", test_cls.shape)

Train top1 rows: (359744, 18)
Valid top1 rows: (44968, 18)
Test top1 rows : (44968, 18)


In [15]:
# =========================================
# 15. SHOW RANKING METRICS
# =========================================
for k in [3, 5, 10]:
    print(f"\n===== Ranking Metrics at K = {k} =====")
    print("Train:", evaluate_grouped_ranking(train_meta, k=k))
    print("Valid:", evaluate_grouped_ranking(valid_meta, k=k))
    print("Test :", evaluate_grouped_ranking(test_meta, k=k))


===== Ranking Metrics at K = 3 =====
Train: {'Precision@3': 0.26660347358121333, 'Recall@3': 0.79981042074364, 'NDCG@3': 0.6805241820380803}
Valid: {'Precision@3': 0.25627853881278534, 'Recall@3': 0.7688356164383562, 'NDCG@3': 0.6444854500161674}
Test : {'Precision@3': 0.25962165688193084, 'Recall@3': 0.7788649706457925, 'NDCG@3': 0.6536454107439945}

===== Ranking Metrics at K = 5 =====
Train: {'Precision@5': 0.1841059197651664, 'Recall@5': 0.9205295988258317, 'NDCG@5': 0.7303562272985135}
Valid: {'Precision@5': 0.18160469667318985, 'Recall@5': 0.9080234833659491, 'NDCG@5': 0.7018862283111399}
Test : {'Precision@5': 0.1802837573385519, 'Recall@5': 0.9014187866927593, 'NDCG@5': 0.7042181267229491}

===== Ranking Metrics at K = 10 =====
Train: {'Precision@10': 0.09985934442270063, 'Recall@10': 0.9985934442270059, 'NDCG@10': 0.7564392458043019}
Valid: {'Precision@10': 0.09980430528375736, 'Recall@10': 0.9980430528375733, 'NDCG@10': 0.7320836140516208}
Test : {'Precision@10': 0.099853228

In [16]:
# =========================================
# 16. SHOW AUXILIARY CLASSIFICATION METRICS
# =========================================
print("Train auxiliary metrics:", classification_metrics(train_cls))
print("Valid auxiliary metrics:", classification_metrics(valid_cls))
print("Test auxiliary metrics :", classification_metrics(test_cls))

Train auxiliary metrics: {'accuracy': 0.9120318893435332, 'precision': 0.5161753913894325, 'recall': 0.5161753913894325, 'f1_score': 0.5161753913894325}
Valid auxiliary metrics: {'accuracy': 0.9044209215442092, 'precision': 0.4743150684931507, 'recall': 0.4743150684931507, 'f1_score': 0.4743150684931507}
Test auxiliary metrics : {'accuracy': 0.9057107276285359, 'precision': 0.48140900195694714, 'recall': 0.48140900195694714, 'f1_score': 0.48140900195694714}


In [17]:
# =========================================
# 17. FINAL SUMMARY TABLE
# =========================================
summary_rows = []

for split_name, df_scored, cls_df in [
    ("Train", train_meta, train_cls),
    ("Valid", valid_meta, valid_cls),
    ("Test", test_meta, test_cls)
]:
    rank5 = evaluate_grouped_ranking(df_scored, k=5)
    rank10 = evaluate_grouped_ranking(df_scored, k=10)
    cls = classification_metrics(cls_df)

    summary_rows.append({
        "Split": split_name,
        "Precision@5": rank5["Precision@5"],
        "Recall@5": rank5["Recall@5"],
        "NDCG@5": rank5["NDCG@5"],
        "Precision@10": rank10["Precision@10"],
        "Recall@10": rank10["Recall@10"],
        "NDCG@10": rank10["NDCG@10"],
        "Accuracy": cls["accuracy"],
        "Precision_cls": cls["precision"],
        "Recall_cls": cls["recall"],
        "F1_score": cls["f1_score"]
    })

summary_df = pd.DataFrame(summary_rows)

display(summary_df)

,Split,Precision@5,Recall@5,NDCG@5,Precision@10,Recall@10,NDCG@10,Accuracy,Precision_cls,Recall_cls,F1_score
0,Train,0.184106,0.920530,0.730356,0.099859,0.998593,0.756439,0.912032,0.516175,0.516175,0.516175
1,Valid,0.181605,0.908023,0.701886,0.099804,0.998043,0.732084,0.904421,0.474315,0.474315,0.474315
2,Test,0.180284,0.901419,0.704218,0.099853,0.998532,0.736736,0.905711,0.481409,0.481409,0.481409


In [18]:
# =========================================
# 18. SAVE RESULT SUMMARY CSV
# =========================================
SUMMARY_CSV = MODEL_DIR / "final_result_summary.csv"
summary_df.to_csv(SUMMARY_CSV, index=False)

print("Saved:", SUMMARY_CSV)

Saved: C:\D drive\item_recommendation\data\model_outputs\final_result_summary.csv


In [19]:
# =========================================
# 19. SHORT FINAL PRINT
# =========================================
print("\n========== FINAL MODEL RESULT SUMMARY ==========")

print("\n[Ranking Metrics at 10]")
print("Train:", evaluate_grouped_ranking(train_meta, k=10))
print("Valid:", evaluate_grouped_ranking(valid_meta, k=10))
print("Test :", evaluate_grouped_ranking(test_meta, k=10))

print("\n[Auxiliary Classification Metrics]")
print("Train:", classification_metrics(train_cls))
print("Valid:", classification_metrics(valid_cls))
print("Test :", classification_metrics(test_cls))

print("\nSaved summary file:")
print(SUMMARY_CSV)


========== FINAL MODEL RESULT SUMMARY ==========

[Ranking Metrics at 10]
Train: {'Precision@10': 0.09985934442270063, 'Recall@10': 0.9985934442270059, 'NDCG@10': 0.7564392458043019}
Valid: {'Precision@10': 0.09980430528375736, 'Recall@10': 0.9980430528375733, 'NDCG@10': 0.7320836140516208}
Test : {'Precision@10': 0.09985322896281802, 'Recall@10': 0.9985322896281801, 'NDCG@10': 0.7367355963550212}

[Auxiliary Classification Metrics]
Train: {'accuracy': 0.9120318893435332, 'precision': 0.5161753913894325, 'recall': 0.5161753913894325, 'f1_score': 0.5161753913894325}
Valid: {'accuracy': 0.9044209215442092, 'precision': 0.4743150684931507, 'recall': 0.4743150684931507, 'f1_score': 0.4743150684931507}
Test : {'accuracy': 0.9057107276285359, 'precision': 0.48140900195694714, 'recall': 0.48140900195694714, 'f1_score': 0.48140900195694714}

Saved summary file:
C:\D drive\item_recommendation\data\model_outputs\final_result_summary.csv


In [20]:
# =========================================
# 20. OPTIONAL FEATURE IMPORTANCE
# =========================================
if hasattr(ranker, "feature_importances_"):
    feature_importance_df = pd.DataFrame({
        "feature": X_train_sorted.columns,
        "importance": ranker.feature_importances_
    }).sort_values("importance", ascending=False)

    display(feature_importance_df.head(20))
else:
    print("Feature importance not available")

,feature,importance
48,candidate_category_Personal-Care-Bath,0.047126
49,candidate_category_Personal-Care-Cosmetics,0.046810
38,candidate_category_Meat-Fresh,0.045158
50,candidate_category_Personal-Care-Hair,0.037961
30,candidate_category_Fruits-Fresh,0.033587
37,candidate_category_Instant-Food,0.033158
26,candidate_category_Dairy-Other,0.031647
44,candidate_category_Pantry-Oils,0.031475
51,candidate_category_Personal-Care-Oral,0.029536
58,candidate_category_Veg-Cooking,0.028652
